In [1]:
import torch.utils.data as data
import numpy as np
import os
import sys
from PIL import Image
import torch
import random
import math
from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F

from torch.autograd import Variable
from torchinfo import summary

print(torchvision.__version__)
print(torch.__version__)

0.10.0+cu111
1.9.0+cu111


In [2]:
torch.cuda.current_device()
[torch.cuda.get_device_name(idx) for idx in range(torch.cuda.device_count())]

['GeForce RTX 3070']

In [27]:
LABEL_NUM = 3

RAW_FRAME_LOC = "/scratch1/zeyut/eat_detection/VideoData_rawFrames/"  
FRAME_LOC = "/scratch1/zeyut/eat_detection/VideoData_torch_224/"  
WIDTH = 224
HEIGHT = 224
CHANNEL = 3
LABEL_NUM = 3
LABEL_TABLE = {"bite": 0, "drink": 1, "non_intake": 2}

seq_len = 2
stride = 8
model_type = 1
batch_size = 16
train_video_list = ["p207_c3","p176_c1","p179_c3","p177_c2","p176_c2"]


In [28]:
train_video_list = [f for f in os.listdir(FRAME_LOC+"train_set") if f.startswith("p")]

In [29]:
weights = []
weight_type =5
class_counts =  [333634, 136993, 1964285]
class_counts = np.array(class_counts)
a = 10000
total = sum(class_counts)
for i in range(LABEL_NUM):
    if class_counts[i] == 0:
        ret.append(0)
    else:
        if weight_type == 1:    
            weights.append(1/LABEL_NUM)
        elif weight_type == 2:
            weights.append(a/(1+LABEL_NUM*class_counts[i]/total))
        elif weight_type == 3:
            weights.append(a/class_counts[i])
        elif weight_type == 4:
            weights.append(a/class_counts[i]**0.5)
        elif weight_type == 5:
            beta = 0.99999
            weights.append((1-beta)/(1-beta**class_counts[i]))
weights = np.array(weights) 
#weights = weights/np.sum(weights)     
weights

array([1.03687791e-05, 1.34070364e-05, 1.00000000e-05])

In [30]:
class FrameSequenceDataset(data.Dataset):
    def __init__(self,root_path,video_list,seq_len,stride,model_type,transform,test_mode=False):
        'Initialization'
        self.root_path = root_path
        self.model_type = model_type
        self.transform = transform
        self.test_mode = test_mode
        self.sample_list = []
        self.label_list = []
        self._get_data_list(video_list, seq_len, stride, model_type)
        
    def __len__(self):
        return len(self.sample_list)

    def __getitem__(self, idx):
        frame_list, labels = self.sample_list[idx], self.label_list[idx]
        frames = self._get_frames(frame_list)
        frames = torch.stack([transforms.functional.to_tensor(frame) for frame in frames])
        if not self.test_mode and self.transform is not None:
            frames = self.transform(frames)
        return frames, labels

    def _get_frames(self, frame_list):
        frames = []
        for frame_loc in frame_list:
            frames.append(Image.open(frame_loc).convert('RGB'))
        return frames

    def _get_data_list(self, video_list, seq_len, stride, model_type):
        for video in video_list:
            frame_locs = []
            frame_labels = []
            f = open(self.root_path + video + "/gt_frame_3labels.txt","r")
            gt_frame = [str.split(line, "\t") for line in f.readlines()]
            for frame_info in gt_frame:
                frame_locs.append(self.root_path + video + "/" + frame_info[0])
                cur_label_idx = LABEL_TABLE[str.split(frame_info[1], "\n")[0]]
                frame_labels.append(cur_label_idx)
            for i in range(0, len(frame_locs)-seq_len, stride):
                self.sample_list.append(frame_locs[i:i+seq_len])
                if model_type == 1:
                    self.label_list.append(np.array(frame_labels[i:i+seq_len]))
                elif model_type == 2:
                    self.label_list.append(np.array(frame_labels[i+seq_len-1]))
        self.label_list = np.array(self.label_list)
        self.sample_list = np.array(self.sample_list)
def denormalize(video_tensor):
    """
    Undoes mean/standard deviation normalization, zero to one scaling,
    and channel rearrangement for a batch of images.
    """
    inverse_normalize = transforms.Normalize(
            mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
            std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
    )
    return (inverse_normalize(video_tensor) * 255.).type(torch.uint8).permute(0, 2, 3, 1).numpy()


In [31]:
preprocess = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5), 
            transforms.ColorJitter(brightness=0.4),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
dataset = FrameSequenceDataset(
        root_path=FRAME_LOC+"train_set/",
        video_list=train_video_list,
        seq_len=seq_len,
        stride=stride,
        model_type=model_type,
        transform=preprocess,
        test_mode=False
        )
dataloader = data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

In [32]:
transform = preprocess

In [33]:
class TimeDistributed(nn.Module):
    def __init__(self, module, batch_first=True):
        super(TimeDistributed, self).__init__()
        self.module = module
        self.batch_first = batch_first
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
        input = x.view(batch_size * time_steps, C, H, W)
        output = self.module(input)
        output = output.view(batch_size, time_steps, -1)
        if self.batch_first is False:
            output = output.permute(1, 0, 2)
        return output

    
class Spatial_Encoder(nn.Module):
    def __init__(self,basemodel='resnet34'):
        super(Spatial_Encoder, self).__init__()
        self._prepare_basemodel(basemodel)
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
        x = x.view(batch_size * time_steps, C, H, W)
        x = self.net(x)
        new_C, new_H, new_W = x.size()[-3:]
        output = x.view(batch_size, time_steps, new_C, new_H, new_W)
        return output
    def _prepare_basemodel(self,basemodel):
        if basemodel == "resnet34":
            model = torchvision.models.resnet34(pretrained=True)
        if basemodel == "resnet50":
            model = torchvision.models.resnet50(pretrained=True)
        module_list = list(model.children())
        del module_list[-1]
        self.net = nn.Sequential(*module_list)

In [34]:
"""
resnet = torchvision.models.resnet34(pretrained=True).cuda()
summary(resnet, input_size=(1, 3, 224, 224))
"""

'\nresnet = torchvision.models.resnet34(pretrained=True).cuda()\nsummary(resnet, input_size=(1, 3, 224, 224))\n'

In [35]:
class RES_LSTM(nn.Module):
    def __init__(self,seq_len=16,basemodel='resnet34'):
        super(RES_LSTM, self).__init__()
        self.encoder = Spatial_Encoder(basemodel)
        if basemodel=='resnet34':
            encoder_size = 512
        if basemodel=='resnet50':
            encoder_size = 2048
        self.lstm = nn.LSTM(input_size=encoder_size,
                            hidden_size=128,
                            num_layers=2,
                            batch_first=True)
        self.batch_norm = nn.BatchNorm1d(num_features=seq_len)
        self.flatten = nn.Flatten(start_dim=2,end_dim=-1)
        self.fc = nn.Sequential(nn.Linear(128, LABEL_NUM),
                                nn.ReLU())
        self.act = nn.Softmax(dim=-1)
    def forward(self, x):
        x = self.encoder(x)
        x = self.flatten(x)
        x = self.batch_norm(x)
        x,(hn, cn) = self.lstm(x)
        x = self.fc(x)
        output = self.act(x)
        return output

In [36]:
model = RES_LSTM(seq_len=seq_len).cuda()
summary(model, input_size=(32,seq_len, CHANNEL, HEIGHT, WIDTH))

Layer (type:depth-idx)                   Output Shape              Param #
├─Spatial_Encoder: 1-1                   [32, 2, 512, 1, 1]        --
|    └─Sequential: 2-1                   [64, 512, 1, 1]           --
|    |    └─Conv2d: 3-1                  [64, 64, 112, 112]        9,408
|    |    └─BatchNorm2d: 3-2             [64, 64, 112, 112]        128
|    |    └─ReLU: 3-3                    [64, 64, 112, 112]        --
|    |    └─MaxPool2d: 3-4               [64, 64, 56, 56]          --
|    |    └─Sequential: 3-5              [64, 64, 56, 56]          221,952
|    |    └─Sequential: 3-6              [64, 128, 28, 28]         1,116,416
|    |    └─Sequential: 3-7              [64, 256, 14, 14]         6,822,400
|    |    └─Sequential: 3-8              [64, 512, 7, 7]           13,114,368
|    |    └─AdaptiveAvgPool2d: 3-9       [64, 512, 1, 1]           --
├─Flatten: 1-2                           [32, 2, 512]              --
├─BatchNorm1d: 1-3                       [32, 2, 512] 

In [37]:
for name, param in model.fc.named_parameters():
    if 'bias' in name:
        nn.init.constant_(param, 0.0)
    elif 'weight' in name:
        nn.init.kaiming_normal_(param)

In [38]:
def class_weights(label_list, weight_type):
    class_counts = []
    for label in range(LABEL_NUM):
        class_counts.append(np.sum(label_list==label))
    class_counts = np.array(class_counts)
    total = np.sum(class_counts)
    ret = []
    for i in range(LABEL_NUM):
        if class_counts[i] == 0:
            ret.append(0)
        else:
            if weight_type == 1:
                """
                version 1: n/(m*c(i))
                where n: total sample number, m: number of classes. c(i): number of samples belonging to the class
                """
                ret.append(total/(class_counts[i] * LABEL_NUM) / np.sum(total/(class_counts[class_counts!=0] * LABEL_NUM)))
            elif weight_type == 2:
                """
                version 2: 1/(1+c(i)/n)
                """
                ret.append(1/(1+LABEL_NUM*class_counts[i]/total) / np.sum(1/(1+LABEL_NUM*class_counts[class_counts!=0]/total)))
            elif weight_type == 3:
                """
                version 3: 1/c(i) / sum(1/c(i)) (Inverse Number of Sample)
                """
                ret.append(1/class_counts[i] / np.sum(1/class_counts[class_counts!=0]))
            elif weight_type == 4:
                """
                version 4: 1/c(i)**0.5 (Inverse of Square Root of Number of Samples)
                """
                ret.append(1/class_counts[i]**0.5 / np.sum(1/class_counts[class_counts!=0]**0.5))
            elif weight_type == 5:
                """
                version 5: (1-beta) / (1-beta**c(i)) (Effective Number of Samples)
                https://medium.com/gumgum-tech/handling-class-imbalance-by-introducing-sample-weighting-in-the-loss-function-3bdebd8203b4
                """
                beta = 0.99999
                ret.append((1-beta)/(1-beta**class_counts[i])/ np.sum((1-beta)/(1-beta**class_counts[class_counts!=0])))
            else:
                ret.append(1/LABEL_NUM)
    return np.array(ret)

In [39]:
for step, (input, target) in enumerate(dataloader):
    input = Variable(input).cuda()
    target = Variable(target).cuda()
    one_hot = F.one_hot(target, num_classes=LABEL_NUM)
    pred = model(input)

    print(pred.size())
    print(input.size())
    print(target.size())
    break

torch.Size([16, 2, 3])
torch.Size([16, 2, 3, 224, 224])
torch.Size([16, 2])


In [46]:
pred = pred.argmax(-1)


In [53]:
pred==target

tensor([[False, False],
        [False, False],
        [False, False],
        [ True,  True],
        [False, False],
        [ True,  True],
        [False, False],
        [ True,  True],
        [False,  True],
        [ True,  True],
        [False, False],
        [False, False],
        [ True,  True],
        [ True,  True],
        [ True,  True],
        [False, False]], device='cuda:0')

In [49]:
target

tensor([[2, 2],
        [2, 2],
        [2, 2],
        [2, 2],
        [2, 2],
        [1, 1],
        [2, 2],
        [2, 2],
        [2, 2],
        [2, 2],
        [2, 2],
        [2, 2],
        [2, 2],
        [2, 2],
        [2, 2],
        [0, 0]], device='cuda:0')

In [55]:
torch.logical_and(pred==target, target==0).sum().item()

0

In [40]:
batch_size = 16
epochs = 5
weights = class_weights(dataset.label_list,1) 
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss(weight=torch.from_numpy(weights))
optimizer = torch.optim.Adam(model.parameters())
model = torch.nn.DataParallel(model).cuda()
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for step, (input, target) in enumerate(dataloader):
        input = Variable(input).cuda()
        target = Variable(target).cuda()
        # Compute prediction and loss
        pred = model(input).permute(0, 2, 1)
        loss = loss_fn(pred, target)
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 10 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= size
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [41]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(dataloader, model, loss_fn, optimizer)
    #test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------


RuntimeError: Expected object of device type cuda but got device type cpu for argument #3 'weight' in call to _thnn_nll_loss2d_forward

In [1]:
checkpoint = torch.load("/home/zeyut/eat_detection/workspace/eating-gesture-detection/model_RES_LSTM_5_16_100_20_v1_torch/"\
                        "checkpoint.tar")

NameError: name 'torch' is not defined

In [ ]:
checkpoint['epoch']

In [42]:
class FrameSequenceDataset(data.Dataset):
    def __init__(self,root_path,video_list,seq_len,stride,model_type,transform,test_mode=False):
        'Initialization'
        self.root_path = root_path
        self.model_type = model_type
        self.transform = transform
        self.test_mode = test_mode
        self.sample_list = []
        self.label_list = []
        self._get_data_list(video_list, seq_len, stride, model_type)
        
    def __len__(self):
        return len(self.sample_list)

    def __getitem__(self, idx):
        frame_list, labels = self.sample_list[idx], self.label_list[idx]
        frames = self._get_frames(frame_list)
        frames = torch.stack([transforms.functional.to_tensor(frame) for frame in frames])
        if not self.test_mode and self.transform is not None:
            frames = self.transform(frames)
        return frames, labels

    def _get_frames(self, frame_list):
        frames = []
        for frame_loc in frame_list:
            frames.append(Image.open(frame_loc).convert('RGB'))
        return frames

    def _get_data_list(self, video_list, seq_len, stride, model_type):
        for video in video_list:
            frame_locs = []
            frame_labels = []
            f = open(os.path.join(self.root_path,video,"gt_frame_3labels.txt"),"r")
            gt_frame = [str.split(line, "\t") for line in f.readlines()]
            for frame_info in gt_frame:
                frame_locs.append(os.path.join(self.root_path,video,frame_info[0]))
                cur_label_idx = LABEL_TABLE[str.split(frame_info[1], "\n")[0]]
                frame_labels.append(cur_label_idx)
            for i in range(0, len(frame_locs)-seq_len, stride):
                self.sample_list.append(frame_locs[i:i+seq_len])
                if model_type == 1:
                    self.label_list.append(np.array(frame_labels[i:i+seq_len]))
                elif model_type == 2:
                    self.label_list.append(np.array(frame_labels[i+seq_len-1]))
        self.label_list = np.array(self.label_list)
        self.sample_list = np.array(self.sample_list)


In [43]:
class testDataset(data.Dataset):
    def __init__(self,data_path,seq_len,stride,transform):
        'Initialization'
        self.data_path = data_path
        self.seq_len = seq_len
        self.stride = stride
        self.transform = transform
        self.input_list = []
        self.video_labels = []
        self.frame_names = []
        self._get_data_list(data_path,seq_len,stride)   
        
    def __len__(self):
        return len(self.input_list)
    
    def __getitem__(self, idx):
        frame_list = self.input_list[idx]
        frames = self._get_frames(frame_list)
        frames = torch.stack([transforms.functional.to_tensor(frame) for frame in frames])
        frames = self.transform(frames)
        return frames
    
    def _get_frames(self, frame_list):
        frames = []
        for frame_loc in frame_list:
            frames.append(Image.open(frame_loc).convert('RGB'))
        return frames
    
    def _get_data_list(self,data_path,seq_len,stride):
        frame_locs = []
        f = open(os.path.join(data_path,"gt_frame_3labels.txt"),"r")
        gt_frame = [str.split(line, "\t") for line in f.readlines()]
        sys.stdout.flush()
        for frame_info in gt_frame:
            self.frame_names.append(frame_info[0])
            frame_locs.append(os.path.join(data_path,frame_info[0]))
            cur_label_idx = LABEL_TABLE[str.split(frame_info[1], "\n")[0]]
            self.video_labels.append(cur_label_idx)
        for i in range(0, len(frame_locs)-seq_len+1, stride):
            self.input_list.append(frame_locs[i:i+seq_len])
        print(len(frame_locs),i)
        self.video_labels = np.array(self.video_labels)       
        self.frame_names = np.array(self.frame_names) 
            

class RateMeter(object):
    """Computes and stores the average rate (acc, TPR, etc)"""
    def __init__(self):
        self.reset()
    def reset(self):
        self.correctCount = 0
        self.totalCount = 0
        self.rate = 0
    def update(self, correct, total):
        self.correctCount += correct
        self.totalCount += total
        if self.totalCount:
            self.rate = self.correctCount / self.totalCount
        else:
            self.rate = 0
            

In [44]:
model_type = 1
test_batch_size=100
test_stride=1
test_video_list = [train_video_list[1]]
model.eval()
preprocess = transforms.Compose([
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                ])


print("{} videos in testing set".format(len(test_video_list)))
print("testing batch size: {}\n".format(test_batch_size))
acc = RateMeter()
total_nonintake = 0
for video_name in test_video_list:
    test_set = testDataset(
                data_path=os.path.join(FRAME_LOC,"train_set",video_name),
                seq_len=seq_len,
                stride=test_stride,
                transform=preprocess
                )
    test_loader = data.DataLoader(
                    dataset=test_set,
                    batch_size=batch_size,
                    shuffle=False,
                    num_workers=10,
                    pin_memory=True
                    )   
    video_labels = test_set.video_labels
    frame_names = test_set.frame_names

    pred_list = []
    prob_list = [] 
    '''
    model_type=1: seq2seq prediction
    get frame wise predictions using max vote strategy
    model_type=2: seq2one prediction
    model directly outputs final frame-wise prediction
    '''    
    with tqdm(test_loader,unit= "batch") as tepoch:
        tepoch.set_description(f"video {video_name}")
        for input in tepoch:
            input = Variable(input).cuda()
            output = model(input)
            cur_prob = output.detach().cpu()
            cur_prob = cur_prob.numpy()
        pred_list.append(cur_prob.argmax(-1))
        prob_list.append(cur_prob)  
    pred_list = np.concatenate(pred_list, axis=0)
    prob_list = np.concatenate(prob_list, axis=0)
    if model_type == 1:   
        # build a heat map
        heat_map = np.zeros((len(video_labels),LABEL_NUM))
        for seq_idx in range(len(pred_list)):
            for frame_idx in range(len(pred_list[seq_idx])):
                heat_map[test_stride*seq_idx+frame_idx][pred_list[seq_idx][frame_idx]] += 1
        video_preds = np.argmax(heat_map, axis=-1)
    elif model_type == 2:
        video_preds = np.zeros(len(video_labels))
        video_preds[seq_len-1:seq_len-1+len(pred_list)] = pred_list  

1 videos in testing set
testing batch size: 100



/home/zeyut/.conda/envs/torch-1.8/lib/python3.8/site-packages/torch/utils/data/dataloader.py:474: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
  0%|          | 0/299 [00:00<?, ?batch/s]

4771 4769


NameError: name 'tbatch' is not defined

In [ ]:
pred_list

In [ ]:
test = np.concatenate(pred_list, axis=0)
test.shape

In [46]:
for input in tepoch:
    print(1)

1
1
1


In [52]:
test_set.video_labels.shape

(4771,)

In [59]:
(video_preds == video_labels).sum()

656

In [121]:
f = open("./test.txt","w")
f.write("test\n")
f.flush()
f.write("test")
f.flush()
f.close()